ChatGroq API

In [2]:
from langchain_groq.chat_models import ChatGroq
import os

# Never hardcode API keys in code.
Groq_Token = os.getenv("CHAT_GROQ_API_KEY")

if not Groq_Token:
    raise ValueError("Set CHAT_GROQ_API_KEY in your environment before running this notebook.")

In [3]:
# Code block to check for available models using the Groq API (from https://console.groq.com/docs/models)
import requests

api_key = Groq_Token
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'whisper-large-v3-turbo', 'object': 'model', 'created': 1728413088, 'owned_by': 'OpenAI', 'active': True, 'context_window': 448, 'public_apps': None, 'max_completion_tokens': 448}, {'id': 'canopylabs/orpheus-v1-english', 'object': 'model', 'created': 1766186316, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'meta-llama/llama-4-scout-17b-16e-instruct', 'object': 'model', 'created': 1743874824, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'qwen/qwen3-32b', 'object': 'model', 'created': 1748396646, 'owned_by': 'Alibaba Cloud', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 40960}, {'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_complet

In [4]:
# Groq LLM model initialization
model_name = "llama-3.1-8b-instant"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

# Statement
sentence = "The product quality is amazing but the delivery was delayed. However I am happy with the customer service."

### Zero Shot

We ask the model to perform a task without providing any examples. The model relies entirely on its pre-trained knowledge and the clarity of the instruction.
- Clear instructions improve output quality
- Works well for simple tasks, may fail on complex tasks

In [5]:
# Zero-shot prompt
query = f"""
You are a sentiment analysis model.
Your task is to analyze the sentiment expressed in the given text and classify it as "positive", "negative", or "neutral".
Provide the sentiment label and a brief reason.

Sentence: {sentence}
"""

answer = llm.invoke(query)
print(answer.content)

Sentiment Label: Neutral

Reason: The text expresses both positive and negative sentiments. The phrase "amazing" indicates a positive sentiment towards the product quality, while the word "delayed" indicates a negative sentiment towards the delivery. However, the overall tone is balanced, and the positive sentiment towards the customer service ("I am happy") outweighs the negative sentiment, resulting in a neutral sentiment.


### Few Shot

We provide a few examples to guide the model. This helps the model understand the expected pattern and improves accuracy.
- Demonstrates input-output format
- Reduces ambiguity

In [6]:
# Few-shot prompt
query = f"""
You are a sentiment analysis model.
Your task is to analyze the sentiment expressed in the given text and classify it as "positive", "negative", or "neutral".
Provide the sentiment label and a brief reason.

Here are a few examples:
1. Sentence: "The customer service was excellent, and I received my order quickly."
Sentiment: Positive

2. Sentence: "The food was bland and the service was slow."
Sentiment: Negative

3. Sentence: "The product is okay, but it's not worth the price."
Sentiment: Neutral

Sentence: {sentence}
"""

answer = llm.invoke(query)
print(answer.content)

Sentiment: Positive

Reason: The text mentions two negative aspects ("the delivery was delayed"), but they are outweighed by two positive aspects ("the product quality is amazing" and "I am happy with the customer service"). The overall tone is positive, indicating that the customer's satisfaction with the product quality and customer service outweighs their dissatisfaction with the delivery delay.


### Role-based prompting

We assign a specific persona or role to the model. This influences tone, style, and sometimes reasoning approach.
- "You are X..." defines behavior
- Useful for creative, domain-specific, or stylistic tasks

In [7]:
model_name = "llama-3.3-70b-versatile"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

In [8]:
# Without persona
query = "Write me a short poem."
answer = llm.invoke(query)
print(answer.content)

Here's a short poem:

The sun sets slow and paints the sky,
A fiery hue that makes me sigh.
The stars come out and twinkle bright,
A night of rest, a peaceful sight.

The world is calm, the darkness deep,
A time for dreams, a time to sleep.
The moon's soft glow, a gentle beam,
Guides me through, a peaceful dream.


In [9]:
# With persona
query = "You are Shakespeare, an English writer. Write me a short poem."
answer = llm.invoke(query)
print(answer.content)

Fairest maiden, with thy beauty's might,
Thou dost enthrall my heart, and senses bright.
As sun doth rise, and night's dark veil doth fade,
Thy lovely face, my thoughts and soul hath made.

Thy eyes, like sapphires, shining bright and blue,
Do sparkle with a fire, that my heart doth renew.
Thy lips, they curve, in sweet and gentle smile,
And in their touch, my love for thee doth pile.

Oh, fairest one, thou art a wondrous dream,
A vision of delight, that doth my soul esteem.
Forever with thee, I wouldst gladly stay,
And in thy love, my heart doth find its way.


### System Prompting

Instead of writing everything in one string, we separate instructions using system and user messages.
- System message defines behavior
- User message defines the task
- More structured and reliable

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant that explains concepts in simple terms."),
    HumanMessage(content="Explain what a black hole is in 2 lines."),
]

response = llm.invoke(messages)
print(response.content)

A black hole is a region in space where gravity is so strong that nothing, including light, can escape once it gets too close. It's formed when a massive star collapses in on itself, creating an incredibly dense point with an intense gravitational pull.


### Structured Reasoning

Structured reasoning asks the model to think step by step before answering. In practice, ask it to reason internally and return a concise final answer plus a brief rationale (instead of full chain-of-thought).
- Encourages logical breakdown
- Improves accuracy on multi-step problems

In [11]:
query = """
Solve the problem.
Think step-by-step internally, then provide:
Final Answer:
Brief Rationale (1 sentence):

Problem: If a train travels 60 km in 1 hour, how far will it travel in 5 hours?
"""

response = llm.invoke(query)
print(response.content)

Final Answer: 300 km
Brief Rationale: To find the distance traveled in 5 hours, we multiply the distance traveled in 1 hour (60 km) by the number of hours (5), resulting in a total distance of 300 km.


### Output Format Control

In [12]:
# JSON
query = """
Extract information from the text and return ONLY valid JSON.

Text: "John bought a laptop for $1200 on 5th May 2026."

Format:
{
  "name": "",
  "item": "",
  "price": "",
  "date": ""
}
"""

response = llm.invoke(query)
print(response.content)

```json
{
  "name": "John",
  "item": "laptop",
  "price": "$1200",
  "date": "5th May 2026"
}
```


In [13]:
# Lists
query = """
List 5 programming languages.

Return as a numbered list only.
"""

response = llm.invoke(query)
print(response.content)

1. Java
2. Python
3. C++
4. JavaScript
5. Ruby


In [14]:
# Structured text
query = """
Summarize the topic "Linked List" in 100 words in this format:

Title:
Definition:
Applications:
Advantages:
"""

response = llm.invoke(query)
print(response.content)

Title: Linked List
Definition: A dynamic data structure consisting of nodes with data and pointers to next nodes.
Applications: Database query results, dynamic memory allocation, and browser history management.
Advantages: Efficient insertion/deletion, flexible memory usage, and good cache performance, making it suitable for applications with frequent data modifications.


### Self-Checking prompts

The model generates an answer, then critiques and improves its own response.
- Introduces self-evaluation
- Often improves quality

In [15]:
def self_checking_prompt(task):
    prompt = f"""
You are an expert assistant.

Step 1: Solve the task.
Step 2: Critically review your answer for correctness, clarity, and completeness.
Step 3: Provide a refined final answer.

Task:
{task}

Return in this format:
Initial Answer:
Critique:
Final Answer:
"""
    response = llm.invoke(prompt)
    return response.content


task = "Compute square root of 16."
print(self_checking_prompt(task))

Initial Answer:
The square root of 16 is 4, since 4 * 4 = 16.

Critique:
The initial answer is correct, but it only provides the positive square root. It's essential to note that the square root of a number can have both a positive and a negative value. In this case, the square root of 16 can be either 4 or -4, as both 4 * 4 = 16 and (-4) * (-4) = 16.

Final Answer:
The square root of 16 is ±4, as both 4 * 4 = 16 and (-4) * (-4) = 16.


### Iterative refinement

The model improves its output over multiple iterations.
- Multi-step improvement loop
- Simulates deeper thinking

In [16]:
def iterative_refinement(task, iterations=3):
    answer = ""

    for _ in range(iterations):
        prompt = f"""
Improve the following answer:

Task: {task}

Current Answer:
{answer}

Provide a better version.
"""
        response = llm.invoke(prompt)
        answer = response.content

    return answer


print(iterative_refinement("Explain dynamic programming in 100 words in simple terms."))

Here's a revised answer in 100 words:

Dynamic programming is a problem-solving approach that breaks down complex problems into smaller parts. It saves solutions to these parts, so you don't repeat calculations. Think of it like a recipe book: once you've made a dish, you can reuse the recipe. This approach helps you solve similar problems quickly, saving time and effort. By building on smaller solutions, dynamic programming efficiently tackles complex challenges, making it a powerful tool for problem-solving. It's a clever way to learn from smaller problems and apply that knowledge to bigger ones.


### Prompt Versioning

Prompts are treated like code and improved over time with version tracking.
- Maintain v1, v2, v3...
- Track performance changes with eval scores

In [32]:
prompt_versions = {
    "v1": """You are a customer support assistant for an e-commerce store.
Respond to the customer request.

Customer message: {input}

Return:
Response:
Next Step:""",
    "v2": """You are a customer support assistant for an e-commerce store.
Guidelines:
- Be empathetic and concise.
- If you need more information, ask one clarifying question.
- Provide a concrete next step.

Customer message: {input}

Return:
Response:
Next Step:""",
    "v3": """You are a customer support assistant for an e-commerce store.
Guidelines:
- Be empathetic and concise (max 3 sentences in Response).
- Acknowledge the issue or request.
- Provide one actionable next step.
- If critical info is missing, ask one clarifying question in Next Step.
- If order-related, request the order number.

Customer message: {input}

Return:
Response:
Next Step:""",
}

# Add v4+ as an exercise and compare scores after evaluation.

### Multi-Example Testing

Test prompts across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases early

In [24]:
test_inputs = [
    "My order #12345 was supposed to arrive yesterday. Can you check the status?",
    "I received a damaged mug. How do I get a replacement?",
    "I want to return a jacket bought last week. What's the return process?",
    "I was charged twice for my order. Please help.",
    "I need to change the shipping address for order #98765.",
    "The promo code SPRING20 isn't working at checkout.",
    "Can I cancel my order? It hasn't shipped yet.",
    "I forgot my password and can't log in.",
    "Do you ship internationally to Canada?",
    "I need an invoice for my last purchase.",
]

### Scoring prompts using LLM

In [20]:
def build_prompt(prompt_or_builder, input_text):
    if callable(prompt_or_builder):
        return prompt_or_builder(input_text)
    return prompt_or_builder.replace("{input}", input_text)


def run_prompt_on_dataset(prompt_or_builder, inputs):
    results = []

    for inp in inputs:
        prompt = build_prompt(prompt_or_builder, inp)
        response = llm.invoke(prompt).content

        results.append({
            "input": inp,
            "output": response
        })

    return results

In [21]:
def llm_judge(input_text, output_text):
    judge_prompt = f"""
Evaluate the response to a customer support request based on:
- Correctness: addresses the request with a plausible action or guidance
- Clarity: concise and easy to follow
- Instruction following: matches any requested format and tone

Score from 1 to 10. Decimals are allowed. Return only a number. No reasoning.

Customer message:
{input_text}

Response:
{output_text}
"""
    score = llm.invoke(judge_prompt).content
    return score

### Testing Harness

Prompts are tested across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases

In [22]:
def test_harness(prompt_or_builder, inputs):
    results = []

    for inp in inputs:
        prompt = build_prompt(prompt_or_builder, inp)
        output = llm.invoke(prompt).content

        score = llm_judge(inp, output)
        results.append({
            "input": inp,
            "output": output,
            "score": score
        })

    return results

In [33]:
prompt_v1 = prompt_versions["v1"]

results = test_harness(prompt_v1, test_inputs)

for r in results:
    print("\n---")
    print("Input:", r["input"])
    print("Score:", r["score"])


---
Input: My order #12345 was supposed to arrive yesterday. Can you check the status?
Score: 8.5

---
Input: I received a damaged mug. How do I get a replacement?
Score: 8.5

---
Input: I want to return a jacket bought last week. What's the return process?
Score: 8.5

---
Input: I was charged twice for my order. Please help.
Score: 8.5

---
Input: I need to change the shipping address for order #98765.
Score: 8.5

---
Input: The promo code SPRING20 isn't working at checkout.
Score: 8.5

---
Input: Can I cancel my order? It hasn't shipped yet.
Score: 8.5

---
Input: I forgot my password and can't log in.
Score: 9.5

---
Input: Do you ship internationally to Canada?
Score: 8.5

---
Input: I need an invoice for my last purchase.
Score: 8.5


### Failure testing

Identify and analyze cases where the model performs poorly.
- Focus on weaknesses, not just averages
- Drives prompt improvement

In [26]:
def get_failures(results, threshold=8.5):
    return [r for r in results if float(r["score"]) < threshold]


failures = get_failures(results)

for f in failures:
    print("\nFAILED CASE:")
    print("Input:", f["input"])
    print("Score:", f["score"])

### Evaluation and Comparison

Compare different prompt versions using scores of outputs.
- Evidence-based improvement
- Use metrics, not intuition

In [34]:
prompt_v2 = prompt_versions["v2"]

results_v2 = test_harness(prompt_v2, test_inputs)

prompt_v3 = prompt_versions["v3"]

results_v3 = test_harness(prompt_v3, test_inputs)

In [35]:
def average_score(results):
    return sum(float(r["score"]) for r in results) / len(results)


print("V1 Avg:", round(average_score(results), 2))
print("V2 Avg:", round(average_score(results_v2), 2))
print("V3 Avg:", round(average_score(results_v3), 2))

V1 Avg: 8.6
V2 Avg: 8.5
V3 Avg: 8.5


In [36]:
version_log = [
    {
        "version": "v1",
        "prompt": prompt_versions["v1"],
        "avg_score": average_score(results),
        "failures": len(get_failures(results)),
        "notes": "Baseline"
    },
    {
        "version": "v2",
        "prompt": prompt_versions["v2"],
        "avg_score": average_score(results_v2),
        "failures": len(get_failures(results_v2)),
        "notes": "Clearer instructions"
    },
    {
        "version": "v3",
        "prompt": prompt_versions["v3"],
        "avg_score": average_score(results_v3),
        "failures": len(get_failures(results_v3)),
        "notes": "Structured constraints"
    },
]

for v in version_log:
    print(v)

{'version': 'v1', 'prompt': 'You are a customer support assistant for an e-commerce store.\nRespond to the customer request.\n\nCustomer message: {input}\n\nReturn:\nResponse:\nNext Step:', 'avg_score': 8.6, 'failures': 0, 'notes': 'Baseline'}
{'version': 'v2', 'prompt': 'You are a customer support assistant for an e-commerce store.\nGuidelines:\n- Be empathetic and concise.\n- If you need more information, ask one clarifying question.\n- Provide a concrete next step.\n\nCustomer message: {input}\n\nReturn:\nResponse:\nNext Step:', 'avg_score': 8.5, 'failures': 0, 'notes': 'Clearer instructions'}
{'version': 'v3', 'prompt': 'You are a customer support assistant for an e-commerce store.\nGuidelines:\n- Be empathetic and concise (max 3 sentences in Response).\n- Acknowledge the issue or request.\n- Provide one actionable next step.\n- If critical info is missing, ask one clarifying question in Next Step.\n- If order-related, request the order number.\n\nCustomer message: {input}\n\nRetur

### Score Summary (Prompt Versions)

| Version | Avg Score | Failures | Notes |
| --- | --- | --- | --- |
| v1 | 8.6 | 0 | Baseline |
| v2 | 8.5 | 0 | Clearer instructions |
| v3 | 8.5 | 0 | Structured constraints |

### Lab: Prompt Engineering Drills

Domain scenario: You are a customer support assistant for an e-commerce store.
Run each prompting pattern below on the same test_inputs list, record outputs, and compare average scores.
Learners should document the best-performing pattern and why it worked.

In [30]:
from langchain_core.messages import SystemMessage, HumanMessage

def build_zero_shot(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Respond to the customer message.

Customer message: {input_text}

Return:
Response:
Next Step:"""


def build_few_shot(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Example 1:
Customer message: "My order is late."
Response: "Sorry about the delay. I can check the status of your order."
Next Step: "Please share your order number."

Example 2:
Customer message: "I received the wrong size."
Response: "I can help with an exchange."
Next Step: "Please confirm your order number and the size you need."

Customer message: {input_text}

Return:
Response:
Next Step:"""


def build_role_based(input_text):
    return f"""You are a senior customer support specialist known for empathy and precision.
Respond to the customer message.

Customer message: {input_text}

Return:
Response:
Next Step:"""


def build_system_prompt(input_text):
    return [
        SystemMessage(content="You are a customer support assistant for an e-commerce store. Be polite, concise, and helpful. Ask one clarifying question if needed."),
        HumanMessage(content=f"Customer message: {input_text}\nReply with Response and Next Step."),
    ]


def build_structured_reasoning(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Think step-by-step internally, then provide:
Final Answer:
Brief Rationale (1 sentence):

Customer message: {input_text}
"""


def build_output_format(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Return ONLY valid JSON with keys:
  "response": "",
  "next_step": "",
  "missing_info": ""

Customer message: {input_text}
"""

In [31]:
pattern_builders = {
    "zero_shot": build_zero_shot,
    "few_shot": build_few_shot,
    "role_based": build_role_based,
    "system_prompt": build_system_prompt,
    "structured_reasoning": build_structured_reasoning,
    "output_format_control": build_output_format,
}

pattern_results = {}
for name, builder in pattern_builders.items():
    pattern_results[name] = test_harness(builder, test_inputs)

for name, results in pattern_results.items():
    print(name, "avg:", round(average_score(results), 2))

best_pattern = max(pattern_results, key=lambda n: average_score(pattern_results[n]))
print("Best pattern:", best_pattern)

zero_shot avg: 8.6
few_shot avg: 8.5
role_based avg: 8.5
system_prompt avg: 8.6
structured_reasoning avg: 8.9
output_format_control avg: 8.5
Best pattern: structured_reasoning


In [37]:
from IPython.display import Markdown, display

lab_scores = [(name, round(average_score(results), 2)) for name, results in pattern_results.items()]
lab_scores_sorted = sorted(lab_scores, key=lambda x: x[1], reverse=True)

table_lines = ["| Pattern | Avg Score |", "| --- | --- |"]
table_lines += [f"| {name} | {score} |" for name, score in lab_scores_sorted]

best = max(lab_scores, key=lambda x: x[1])[0]
display(Markdown("### Lab Score Summary\n\n" + "\n".join(table_lines) + f"\n\nBest pattern: **{best}**"))

### Lab Score Summary

| Pattern | Avg Score |
| --- | --- |
| structured_reasoning | 8.9 |
| zero_shot | 8.6 |
| system_prompt | 8.6 |
| few_shot | 8.5 |
| role_based | 8.5 |
| output_format_control | 8.5 |

Best pattern: **structured_reasoning**